# Day 1b — Baselines: k-mer + classical ML, and one-hot + CNN

Two ways to turn a DNA string into something a model can use, before we ever touch a
pretrained language model:

1. **k-mer frequencies**: count how often each substring of length k (e.g. all 256
   4-mers) appears, normalize -> a fixed-size vector -> classical ML (logistic regression).
2. **One-hot + CNN**: encode each nucleotide as a 4-dim one-hot vector (A/C/G/T), stack
   into a (4, 200) tensor, and let a small 1D CNN learn its own features directly from
   the raw sequence.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import torch
from data import load_all
from featurize import kmer_matrix, one_hot_batch
from models.baselines import make_kmer_classifier, OneHotCNN
from eval import evaluate_sklearn, evaluate_logits, count_params, measure_latency_sklearn, measure_latency_torch

splits = load_all("../../data/supervised/processed")
train, val = splits["train"], splits["val"]

## Baseline 1: k-mer frequencies + logistic regression

In [ ]:
K = 4
X_train_kmer = kmer_matrix(train["sequence"].tolist(), k=K)
X_val_kmer = kmer_matrix(val["sequence"].tolist(), k=K)
y_train, y_val = train["label"].to_numpy(), val["label"].to_numpy()

kmer_clf = make_kmer_classifier("logreg")
kmer_clf.fit(X_train_kmer, y_train)

kmer_metrics = evaluate_sklearn(kmer_clf, X_val_kmer, y_val)
print("k-mer + logreg:", kmer_metrics)

## Baseline 2: one-hot nucleotides + small CNN

In [ ]:
WINDOW = 200
X_train_oh = torch.tensor(one_hot_batch(train["sequence"].tolist(), length=WINDOW))
X_val_oh = torch.tensor(one_hot_batch(val["sequence"].tolist(), length=WINDOW))
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

cnn = OneHotCNN(seq_len=WINDOW)
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)

n = X_train_oh.shape[0]
for epoch in range(5):
    perm = torch.randperm(n)
    epoch_loss = 0.0
    for start in range(0, n, 256):
        idx = perm[start:start + 256]
        optimizer.zero_grad()
        logits = cnn(X_train_oh[idx])
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y_train_t[idx])
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    print(f"epoch {epoch+1}: loss={epoch_loss/n:.4f}")

In [ ]:
with torch.no_grad():
    val_logits = cnn(X_val_oh).numpy()
cnn_metrics = evaluate_logits(val_logits, y_val)
print("one-hot CNN:", cnn_metrics)

## Compare

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {"model": "kmer+logreg", **kmer_metrics,
     "params": X_train_kmer.shape[1] + 1,
     "latency_ms": measure_latency_sklearn(kmer_clf, X_val_kmer[:1])},
    {"model": "onehot+CNN", **cnn_metrics,
     "params": count_params(cnn),
     "latency_ms": measure_latency_torch(cnn, X_val_oh[:1])},
])
comparison

### Checkpoint

You should have an accuracy/F1/params/latency table for both baselines. Save these numbers
(or the `comparison` DataFrame) — they go into the final Day-4 efficiency chart.

Next: `02_evo2_embeddings_and_classifier.ipynb` — does a genomic foundation model beat this?